# BITE spike — first 3D dog mesh (Phase 3, Chunk A)

Throwaway validation, **verbatim from `camenduru/bite-colab`**: a patched `dev`
fork that runs on modern torch (skips the 2020 `pytorch3d 0.2.5` build), with
checkpoint/data bundled in a public HF repo — **no MPI registration**. See
`../SETUP.md`. Runtime ▸ Change runtime type ▸ **GPU**.

## 0 · GPU check

In [ ]:
!nvidia-smi

## 1 · Clone fork + bundled assets, install (camenduru's exact recipe)

In [ ]:
%cd /content
!git clone -b dev https://github.com/camenduru/bite_gradio-hf
!git clone https://huggingface.co/camenduru/bite
!mv /content/bite/checkpoint /content/bite_gradio-hf/checkpoint
!mv /content/bite/data /content/bite_gradio-hf/data
!mv /content/bite/datasets /content/bite_gradio-hf/datasets
!rm -rf /content/bite
%cd /content/bite_gradio-hf
!pip install -r requirements.txt

## 2 · Quickest quality check — Gradio app
Upload each test photo in the UI and eyeball the recovered mesh. Stop the cell
when done; run the batch path below for saveable overlays.

In [ ]:
%cd /content/bite_gradio-hf
!python ./scripts/gradio_demo.py

## 3 · Batch path — crop with YOLO, then run ttopt
BITE has no detector → crop dog-centred first. Upload the spike-set photos.

In [ ]:
!pip -q install ultralytics
from google.colab import files
import os, glob, numpy as np
from PIL import Image
RAW='/content/raw'; CROP='/content/bite_gradio-hf/datasets/test_image_crops'
os.makedirs(RAW,exist_ok=True); os.makedirs(CROP,exist_ok=True)
for n,d in files.upload().items(): open(os.path.join(RAW,n),'wb').write(d)
from ultralytics import YOLO
yolo=YOLO('yolov8n-seg.pt'); M=0.12
for p in sorted(glob.glob(RAW+'/*')):
    im=Image.open(p).convert('RGB'); W,H=im.size
    b=yolo(np.array(im),classes=[16],verbose=False)[0].boxes
    if b is None or len(b)==0: print('no dog',os.path.basename(p)); continue
    i=int(((b.xyxy[:,2]-b.xyxy[:,0])*(b.xyxy[:,3]-b.xyxy[:,1])).argmax())
    x1,y1,x2,y2=[float(v) for v in b.xyxy[i].cpu().numpy()]; bw,bh=x2-x1,y2-y1
    c=im.crop((int(max(0,x1-bw*M)),int(max(0,y1-bh*M)),int(min(W,x2+bw*M)),int(min(H,y2+bh*M))))
    o=os.path.join(CROP,os.path.splitext(os.path.basename(p))[0]+'.png'); c.save(o); print('cropped',os.path.basename(o),c.size)
print('crops:',os.listdir(CROP))

In [ ]:
import time
%cd /content/bite_gradio-hf
t0=time.time()
!python scripts/full_inference_including_ttopt.py --workers 4 \
  --config refinement_cfg_test_withvertexwisegc_csaddnonflat_crops.yaml \
  --model-file-complete cvpr23_dm39dnnv3barcv2b_refwithgcpervertisflat0morestanding0/checkpoint.pth.tar \
  --suffix ttopt_spike
print('TOTAL %.1fs (record per-image timing for Chunk B)'%(time.time()-t0))

## 4 · View + save overlays → download into `dog-pose/bite_server/spike_outputs/`

In [ ]:
import glob,os
from IPython.display import Image as IPy, display
pngs=sorted(glob.glob('/content/bite_gradio-hf/results/**/*.png',recursive=True),key=os.path.getmtime,reverse=True)
print('result pngs:',len(pngs)); [display(IPy(filename=p)) for p in pngs[:8]]
!cd /content/bite_gradio-hf && zip -q -r /content/bite_spike_outputs.zip results; echo zipped
from google.colab import files; files.download('/content/bite_spike_outputs.zip')

---
**If the `dev` fork breaks**, fall back to the faithful 2020 conda env (see
`../SETUP.md`). If that also stalls within a time-box, verdict is "BITE not
viable as-is" → cloud/Docker, or ship Plan Chunk E (richer 2D + Rough.js) alone.